# Módulo 2: Armazenamento, Data Lake e ELT (Exercícios)

Cinco exercícios práticos e simplificados, cobrindo os quatro tópicos principais do módulo:

1. **Comparativo CSV vs. Parquet** (exercício 1)
2. **Data Lake e Arquitetura Medalhão: Bronze e Silver** (exercício 2)
3. **Camada Gold: indicadores de negócio** (exercício 3)
4. **Consultas OLAP com DuckDB sobre Parquet** (exercício 4)
5. **Padrão ELT com schemas no DuckDB** (exercício 5)

Cada exercício tem uma célula para a sua resposta e, logo depois, uma célula **"Confira sua resposta"**. Se ela imprimir `✅`, a resposta está certa; se der erro, revise o código.

Os arquivos de entrada e as pastas criadas por você ficam isolados dentro de `../../../data/modulo_02/exercicios/`, sem alterar os dados dos outros notebooks.

In [ ]:
# Se estiver em um ambiente novo, instale as dependências descomentando a linha abaixo:
# %pip install pandas duckdb pyarrow numpy

import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb

PASTA_EX = Path("../../../data/modulo_02/exercicios")
PASTA_EX.mkdir(parents=True, exist_ok=True)

print("Ambiente pronto! Arquivos de entrada disponíveis:")
for arq in sorted(PASTA_EX.glob("*.csv")):
    print(" -", arq)

## Parte 1: Formatos de armazenamento (CSV vs. Parquet)

### Exercício 1: medindo tamanho em disco e tempo de leitura

O código abaixo gera `df_teste` com **100 mil registros** fictícios. Salve o mesmo DataFrame em dois formatos e compare espaço e velocidade:

- Salve em CSV no caminho `../../../data/modulo_02/exercicios/comparativo.csv` (`index=False`);
- Salve em Parquet no caminho `../../../data/modulo_02/exercicios/comparativo.parquet` (`engine="pyarrow"`, `index=False`);
- Calcule `tamanho_csv_mb` e `tamanho_parquet_mb` usando `os.path.getsize(...) / (1024 * 1024)`;
- Cronometre a leitura de volta com `time.time()`: guarde os tempos em `tempo_leitura_csv` e `tempo_leitura_parquet`, e guarde o DataFrame lido do Parquet em `df_lido_parquet`.

**Para discutir:** por que o arquivo Parquet ocupa bem menos espaço em disco mesmo armazenando exatamente os mesmos dados do CSV?

In [ ]:
# Base de teste gerada em memória (não altere esta parte)
np.random.seed(42)
n_linhas = 100_000
df_teste = pd.DataFrame({
    "id_pedido": range(n_linhas),
    "categoria": np.random.choice(["Bebidas", "Salgados", "Doces", "Cafés"], n_linhas),
    "valor": np.round(np.random.uniform(4.0, 80.0, n_linhas), 2),
    "status": np.random.choice(["Concluído", "Pendente", "Cancelado"], n_linhas),
})

caminho_csv = "../../../data/modulo_02/exercicios/comparativo.csv"
caminho_parquet = "../../../data/modulo_02/exercicios/comparativo.parquet"

# 1. Salve df_teste em CSV e em Parquet
# escreva seu código aqui

# 2. Calcule o tamanho em MB de cada arquivo
tamanho_csv_mb = 0.0
tamanho_parquet_mb = 0.0

# 3. Meça o tempo de leitura de cada formato
tempo_leitura_csv = 0.0
tempo_leitura_parquet = 0.0
df_lido_parquet = None

print(f"CSV:     {tamanho_csv_mb:.2f} MB | leitura: {tempo_leitura_csv:.4f}s")
print(f"Parquet: {tamanho_parquet_mb:.2f} MB | leitura: {tempo_leitura_parquet:.4f}s")

In [ ]:
# Confira sua resposta (não altere esta célula)
assert os.path.exists(caminho_csv), "O arquivo CSV não foi criado em ../../../data/modulo_02/exercicios/comparativo.csv"
assert os.path.exists(caminho_parquet), "O arquivo Parquet não foi criado em ../../../data/modulo_02/exercicios/comparativo.parquet"
assert tamanho_csv_mb > 0 and tamanho_parquet_mb > 0, "Calcule os tamanhos em MB usando os.path.getsize"
assert tamanho_parquet_mb < tamanho_csv_mb, "O arquivo Parquet deveria ser menor que o CSV"
assert tempo_leitura_csv > 0 and tempo_leitura_parquet > 0, "Cronometre os tempos de leitura com time.time()"
assert df_lido_parquet is not None and len(df_lido_parquet) == 100_000, "df_lido_parquet precisa ter 100.000 linhas"
print(f"✅ Exercício 1 correto (Parquet {tamanho_csv_mb / tamanho_parquet_mb:.1f}x menor!)")

## Parte 2: Data Lake e Arquitetura Medalhão

### Exercício 2: construindo as camadas Bronze e Silver

Simule um Data Lake local para uma cafeteria dentro de `../../../data/modulo_02/exercicios/datalake/` criando as pastas `bronze`, `silver` e `gold`:

1. **Camada Bronze:** leia `../../../data/modulo_02/exercicios/pedidos_bronze.csv` e salve uma cópia fiel (sem alterações) em `../../../data/modulo_02/exercicios/datalake/bronze/pedidos_raw.csv` (`index=False`).
2. **Camada Silver:** a partir da Bronze, faça a limpeza básica:
   - remova as linhas com `data` vazia (`dropna(subset=["data"])`);
   - remova espaços extras e deixe `produto` em minúsculas (`.str.strip().str.lower()`);
   - remova espaços extras de `categoria` (`.str.strip()`).
3. Salve o resultado limpo em Parquet no caminho `../../../data/modulo_02/exercicios/datalake/silver/pedidos_cleaned.parquet` (`engine="pyarrow"`, `index=False`).

*Dica:* use `Path(pasta).mkdir(parents=True, exist_ok=True)` para criar as três pastas.

In [ ]:
# 1. Crie as pastas do Data Lake: bronze, silver e gold
# escreva seu código aqui

# 2. Leia ../../../data/modulo_02/exercicios/pedidos_bronze.csv e salve na camada Bronze
# escreva seu código aqui

# 3. Limpe os dados e salve na camada Silver em Parquet
# df_silver = ...

display(df_silver)

In [ ]:
# Confira sua resposta (não altere esta célula)
for camada in ["bronze", "silver", "gold"]:
    assert Path(f"../../../data/modulo_02/exercicios/datalake/{camada}").is_dir(), f"Faltou criar a pasta {camada}"

assert Path("../../../data/modulo_02/exercicios/datalake/bronze/pedidos_raw.csv").exists(), "Faltou salvar pedidos_raw.csv na Bronze"
caminho_silver = Path("../../../data/modulo_02/exercicios/datalake/silver/pedidos_cleaned.parquet")
assert caminho_silver.exists(), "Faltou salvar pedidos_cleaned.parquet na Silver"

df_s = pd.read_parquet(caminho_silver)
assert len(df_s) == 8, f"Após remover datas vazias deveriam restar 8 linhas, obtido {len(df_s)}"
assert df_s["data"].isna().sum() == 0, "Ainda existem datas nulas na camada Silver"
produtos_esperados = {"cafe expresso", "pao de queijo", "bolo de cenoura", "agua mineral", "croissant"}
assert set(df_s["produto"]) == produtos_esperados, f"Produtos não estão padronizados: {set(df_s['produto'])}"
print("✅ Exercício 2 correto")

### Exercício 3: agregando indicadores na camada Gold

Leia o arquivo limpo da camada Silver (`../../../data/modulo_02/exercicios/datalake/silver/pedidos_cleaned.parquet`) e gere uma tabela agregada por produto para o time de gestão:

- Agrupe por `produto` e calcule:
  - `qtd_pedidos`: contagem de `id_pedido`;
  - `faturamento_total`: soma de `valor`;
- Ordene do maior para o menor `faturamento_total`;
- Salve em `../../../data/modulo_02/exercicios/datalake/gold/faturamento_produto.parquet` (`engine="pyarrow"`, `index=False`).

*Dica:* `df.groupby("produto", as_index=False).agg(qtd_pedidos=("id_pedido", "count"), faturamento_total=("valor", "sum"))`.

In [ ]:
# Leia da Silver, agregue os indicadores e salve na camada Gold
# df_gold = ...

display(df_gold)

In [ ]:
# Confira sua resposta (não altere esta célula)
caminho_gold = Path("../../../data/modulo_02/exercicios/datalake/gold/faturamento_produto.parquet")
assert caminho_gold.exists(), "Faltou salvar faturamento_produto.parquet na Gold"

df_g = pd.read_parquet(caminho_gold)
assert {"produto", "qtd_pedidos", "faturamento_total"}.issubset(df_g.columns), \
    "A tabela Gold deve conter as colunas produto, qtd_pedidos e faturamento_total"
assert len(df_g) == 5, f"Deveriam existir 5 produtos na Gold, obtido {len(df_g)}"
assert round(df_g["faturamento_total"].sum(), 2) == 63.0, "O faturamento total deveria ser 63.0"
cafe = df_g.set_index("produto").loc["cafe expresso"]
assert cafe["qtd_pedidos"] == 3 and cafe["faturamento_total"] == 25.5, \
    "cafe expresso deveria ter 3 pedidos e faturamento_total = 25.5"
print("✅ Exercício 3 correto")

## Parte 3: Consultas OLAP com DuckDB sobre Parquet

### Exercício 4: SQL direto nos arquivos do Data Lake

Sem criar tabelas nem carregar o arquivo com o Pandas, use o **DuckDB** para consultar diretamente `'../../../data/modulo_02/exercicios/datalake/silver/pedidos_cleaned.parquet'` e calcular o resumo diário de vendas:

- Colunas esperadas: `data`, `qtd_pedidos` e `total_dia`;
- Ordene os resultados por `data` em ordem crescente;
- Guarde o DataFrame resultante em `df_por_dia`.

*Dica:* `duckdb.query(sql_ex4).df()` permite apontar `FROM '../../../data/modulo_02/exercicios/datalake/silver/pedidos_cleaned.parquet'` diretamente no SQL.

In [ ]:
sql_ex4 = """
-- escreva sua consulta SQL aqui
"""

df_por_dia = duckdb.query(sql_ex4).df()
display(df_por_dia)

In [ ]:
# Confira sua resposta (não altere esta célula)
assert {"data", "qtd_pedidos", "total_dia"}.issubset(df_por_dia.columns), \
    "A consulta deve retornar as colunas data, qtd_pedidos e total_dia"
assert len(df_por_dia) == 4, f"Deveriam existir 4 datas distintas, obtido {len(df_por_dia)}"
assert round(df_por_dia["total_dia"].sum(), 2) == 63.0, "A soma de total_dia deveria ser 63.0"
por_dia = df_por_dia.set_index("data")
assert por_dia.loc["2026-05-01", "qtd_pedidos"] == 2 and por_dia.loc["2026-05-01", "total_dia"] == 14.5, \
    "Em 2026-05-01 deveriam existir 2 pedidos somando 14.5"
assert por_dia.loc["2026-05-02", "qtd_pedidos"] == 2 and por_dia.loc["2026-05-02", "total_dia"] == 20.5, \
    "Em 2026-05-02 deveriam existir 2 pedidos somando 20.5"
print("✅ Exercício 4 correto")

## Parte 4: Padrão ELT e schemas no DuckDB

### Exercício 5: pipeline ELT com schemas `raw`, `staging` e `analytics`

No padrão **ELT**, carregamos primeiro os dados brutos no banco e fazemos as transformações usando SQL dentro da engine.

O arquivo `../../../data/modulo_02/exercicios/clientes_cadastro.csv` já será carregado para você em `raw.clientes`. Escreva as duas queries SQL abaixo:

1. **`staging.clientes_tratados`:** a partir de `raw.clientes`, remova duplicados (`SELECT DISTINCT`), descarte linhas onde `nome IS NULL`, padronize `nome` em maiúsculas sem espaços extras (`UPPER(TRIM(nome))`), remova espaços de `cidade` (`TRIM(cidade)`), e converta `'N/A'` em `NULL` transformando `idade` em `INTEGER`.
2. **`analytics.clientes_perfil`:** a partir de `staging.clientes_tratados`, mantenha `id_cliente`, `nome`, `idade`, `cidade` e crie a coluna `faixa_etaria`:
   - `'Jovem'` quando `idade < 30`;
   - `'Adulto'` quando `idade >= 30`;
   - `'Não Informado'` caso contrário.

*Dica:* use `TRY_CAST(idade AS INTEGER)` (ou `CAST(CASE WHEN idade = 'N/A' THEN NULL ELSE idade END AS INTEGER)`).

**Para discutir:** qual é a diferença prática entre tratar os dados em Python antes de salvar (ETL, como na Parte 2) e carregar tudo em `raw` para transformar com SQL no banco (ELT, como aqui)?

In [ ]:
con = duckdb.connect(database=":memory:")

con.execute("CREATE SCHEMA IF NOT EXISTS raw;")
con.execute("CREATE SCHEMA IF NOT EXISTS staging;")
con.execute("CREATE SCHEMA IF NOT EXISTS analytics;")

# Load: carrega o CSV bruto direto para o schema raw
con.execute("""
    CREATE OR REPLACE TABLE raw.clientes AS
    SELECT * FROM read_csv_auto('../../../data/modulo_02/exercicios/clientes_cadastro.csv')
""")

print("Camada RAW:")
display(con.execute("SELECT * FROM raw.clientes").df())

# Transform (parte 1): preencha a query para criar staging.clientes_tratados
sql_staging = """
-- CREATE OR REPLACE TABLE staging.clientes_tratados AS ...
"""

# Transform (parte 2): preencha a query para criar analytics.clientes_perfil
sql_analytics = """
-- CREATE OR REPLACE TABLE analytics.clientes_perfil AS ...
"""

con.execute(sql_staging)
con.execute(sql_analytics)

print("Camada ANALYTICS:")
display(con.execute("SELECT * FROM analytics.clientes_perfil ORDER BY id_cliente").df())

In [ ]:
# Confira sua resposta (não altere esta célula)
df_raw = con.execute("SELECT * FROM raw.clientes").df()
df_stg = con.execute("SELECT * FROM staging.clientes_tratados").df()
df_ana = con.execute("SELECT * FROM analytics.clientes_perfil").df()

assert len(df_raw) == 7, "raw.clientes deveria manter as 7 linhas originais"
assert len(df_stg) == 5, f"staging.clientes_tratados deveria ter 5 linhas únicas e válidas, obtido {len(df_stg)}"
assert df_stg["nome"].isna().sum() == 0, "Não deveria haver nome nulo em staging.clientes_tratados"
assert df_stg["nome"].str.isupper().all(), "Todos os nomes em staging devem estar em maiúsculas"
assert df_stg["idade"].isna().sum() == 1, "A idade 'N/A' deveria ter virado NULL em staging"

assert {"id_cliente", "nome", "idade", "cidade", "faixa_etaria"}.issubset(df_ana.columns), \
    "Faltam colunas em analytics.clientes_perfil"
contagem = df_ana["faixa_etaria"].value_counts().to_dict()
assert contagem == {"Jovem": 2, "Adulto": 2, "Não Informado": 1}, \
    f"Distribuição esperada de faixa_etaria: Jovem=2, Adulto=2, Não Informado=1. Obtido: {contagem}"
print("✅ Exercício 5 correto")

In [ ]:
con.close()
print("Fim dos exercícios do Módulo 2.")